In [1]:
pip install -q sentence-transformers xgboost scikit-learn pandas numpy ipywidgets matplotlib

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, accuracy_score, classification_report
from sentence_transformers import SentenceTransformer
import xgboost as xgb
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

RANDOM_SEED = 42
CSV_PATH = "fashion_master.csv"  

In [2]:
df = pd.read_csv(CSV_PATH)
print(df.shape)
df.head()

(6000, 17)


,product_id,idea_text,gender,category,subcategory,fabric,theme,season,occasion,color,source_site,price,rating,review_count,scrape_date,success_score,success_label
0,P000000,"gothic inspired sunglasses in maroon leather, ...",male,accessories,sunglasses,leather,gothic,all-season,casual daily wear,maroon,synthetic_v1,458.26,4.1,468,29-06-2026,63.09,medium
1,P000001,Minimal beige kurta crafted in organic cotton ...,male,ethnic wear,kurta,organic cotton,gothic,spring,wedding guest,beige,synthetic_v1,1552.82,4.6,184,29-06-2026,62.78,low
2,P000002,monsoon edit inspired maxi dress in charcoal g...,female,dresses,maxi dress,velvet,monsoon edit,summer,date night,charcoal grey,synthetic_v1,1352.11,3.7,275,29-06-2026,61.38,low
3,P000003,Designing a navy blue knit salwar suit with a ...,female,ethnic wear,salwar suit,knit,formal office,autumn,office wear,navy blue,synthetic_v1,1792.36,4.2,102,29-06-2026,62.97,low
4,P000004,A korean fashion navy blue scarf for unisex de...,unisex,accessories,scarf,recycled polyester,korean fashion,winter,beach wear,navy blue,synthetic_v1,2403.18,3.9,48,29-06-2026,64.02,medium


In [3]:
TEXT_COL = "idea_text"
CAT_COLS = ["gender", "category", "subcategory", "fabric", "theme", "season", "occasion", "color"]
NUM_COLS = ["price", "rating", "review_count"]
REG_TARGET = "success_score"
CLF_TARGET = "success_label"

# Build lookup of valid values per categorical column -- the interactive
# form's dropdowns are generated from this, so they always match training data.
CATEGORY_OPTIONS = {col: sorted(df[col].dropna().unique().tolist()) for col in CAT_COLS}
CATEGORY_OPTIONS


{'gender': ['female', 'male', 'unisex'],
 'category': ['accessories',
  'activewear',
  'bottomwear',
  'dresses',
  'ethnic wear',
  'footwear',
  'innerwear',
  'outerwear',
  'topwear'],
 'subcategory': ['bag',
  'belt',
  'blazer',
  'blouse',
  'bomber jacket',
  'boots',
  'boxers',
  'bra',
  'briefs',
  'cap',
  'cargo pants',
  'coat',
  'crop top',
  'denim jacket',
  'flats',
  'gown',
  'gym shorts',
  'heels',
  'hoodie',
  'jacket',
  'jeans',
  'jewelry',
  'joggers',
  'kurta',
  'kurti',
  'leggings',
  'lehenga',
  'loafers',
  'maxi dress',
  'midi dress',
  'mini dress',
  'polo',
  'running tights',
  'salwar suit',
  'sandals',
  'saree',
  'scarf',
  'sherwani',
  'shift dress',
  'shirt',
  'shorts',
  'skirt',
  'sneakers',
  'sports bra',
  'sunglasses',
  'sweater',
  't-shirt',
  'tank top',
  'thermal wear',
  'tracksuit',
  'trouser',
  'watch',
  'windcheater',
  'wrap dress',
  'yoga pants'],
 'fabric': ['bamboo fabric',
  'chiffon',
  'cotton',
  'denim

In [4]:
import os

EMBED_CACHE_PATH = "embeddings_cache.npy"
bert_model = SentenceTransformer("all-MiniLM-L6-v2")

if os.path.exists(EMBED_CACHE_PATH):
    text_embeddings = np.load(EMBED_CACHE_PATH)
    print("Loaded cached embeddings:", text_embeddings.shape)
else:
    text_embeddings = bert_model.encode(
        df[TEXT_COL].tolist(),
        show_progress_bar=True,
        batch_size=64,
    )
    np.save(EMBED_CACHE_PATH, text_embeddings)
    print("Computed embeddings:", text_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\Vidita Kalra\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Vidita Kalra\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/94 [00:00<?, ?it/s]

Computed embeddings: (6000, 384)


In [5]:
encoders = {}
encoded_cats = np.zeros((len(df), len(CAT_COLS)))
for i, col in enumerate(CAT_COLS):
    le = LabelEncoder()
    encoded_cats[:, i] = le.fit_transform(df[col])
    encoders[col] = le

scaler = StandardScaler()
scaled_nums = scaler.fit_transform(df[NUM_COLS])

label_encoder = LabelEncoder()
y_clf = label_encoder.fit_transform(df[CLF_TARGET])
y_reg = df[REG_TARGET].values

# Final feature matrix: [BERT embeddings | encoded categoricals | scaled numerics]
X = np.hstack([text_embeddings, encoded_cats, scaled_nums])
print("Final X shape:", X.shape)
print("Classes:", label_encoder.classes_)


Final X shape: (6000, 395)
Classes: ['high' 'low' 'medium']


In [6]:
X_train, X_test, yreg_train, yreg_test, yclf_train, yclf_test = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=RANDOM_SEED
)

reg_model = xgb.XGBRegressor(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_SEED,
)
reg_model.fit(X_train, yreg_train)

clf_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_SEED,
)
clf_model.fit(X_train, yclf_train)

# Quick eval
pred_reg = reg_model.predict(X_test)
pred_clf = clf_model.predict(X_test)
print(f"Regression MAE: {mean_absolute_error(yreg_test, pred_reg):.2f}")
print(f"Classification accuracy: {accuracy_score(yclf_test, pred_clf):.3f}")
print()
print(classification_report(yclf_test, pred_clf, target_names=label_encoder.classes_))


Regression MAE: 1.37
Classification accuracy: 0.792

              precision    recall  f1-score   support

        high       0.86      0.80      0.83       410
         low       0.86      0.83      0.85       404
      medium       0.67      0.73      0.70       386

    accuracy                           0.79      1200
   macro avg       0.80      0.79      0.79      1200
weighted avg       0.80      0.79      0.79      1200



In [7]:
def predict_idea(idea_text, gender, category, subcategory, fabric, theme, season, occasion, color,
                  price, rating, review_count):
    # embed the new text
    new_embedding = bert_model.encode([idea_text])

    # encode categoricals -- unseen values fall back to the most common class
    cat_values = [gender, category, subcategory, fabric, theme, season, occasion, color]
    encoded_row = []
    for col, val in zip(CAT_COLS, cat_values):
        le = encoders[col]
        if val in le.classes_:
            encoded_row.append(le.transform([val])[0])
        else:
            encoded_row.append(0)  # unseen category fallback
    encoded_row = np.array(encoded_row).reshape(1, -1)

    scaled_row = scaler.transform(pd.DataFrame([[price, rating, review_count]], columns=NUM_COLS))

    X_new = np.hstack([new_embedding, encoded_row, scaled_row])

    score = float(reg_model.predict(X_new)[0])
    score = float(np.clip(score, 0, 100))
    label_idx = clf_model.predict(X_new)[0]
    label = label_encoder.inverse_transform([label_idx])[0]
    proba = clf_model.predict_proba(X_new)[0]
    proba_dict = dict(zip(label_encoder.classes_, proba))

    # similarity to existing ideas (cosine sim on BERT embeddings)
    norms = text_embeddings / np.linalg.norm(text_embeddings, axis=1, keepdims=True)
    new_norm = new_embedding / np.linalg.norm(new_embedding, axis=1, keepdims=True)
    sims = (norms @ new_norm.T).flatten()
    top_idx = np.argsort(-sims)[:5]
    similar_df = df.iloc[top_idx][[TEXT_COL, REG_TARGET, CLF_TARGET]].copy()
    similar_df["similarity"] = sims[top_idx].round(3)

    return {
        "score": score,
        "label": label,
        "proba": proba_dict,
        "similar": similar_df,
    }


def suggest_improvements(result, fabric, season, theme, subcategory):
    """Simple rule-based suggestions based on what the data shows works best."""
    suggestions = []

    top_themes = df.groupby("theme")[REG_TARGET].mean().sort_values(ascending=False)
    if theme not in top_themes.index[:5]:
        suggestions.append(
            f"Theme '{theme}' scores lower on average. Consider '{top_themes.index[0]}' "
            f"or '{top_themes.index[1]}' instead -- both trend higher in this dataset."
        )

    top_subcats = df.groupby("subcategory")[REG_TARGET].mean().sort_values(ascending=False)
    if subcategory not in top_subcats.index[:8]:
        suggestions.append(
            f"'{subcategory}' is a lower-performing subcategory here. "
            f"'{top_subcats.index[0]}' tends to score higher."
        )

    bad_combo_penalty = {
        ("wool", "summer"), ("fleece", "summer"), ("velvet", "summer"),
        ("silk", "monsoon"), ("denim", "monsoon"),
    }
    if (fabric, season) in bad_combo_penalty:
        suggestions.append(f"'{fabric}' is a mismatch for '{season}' -- this is hurting your score.")

    if not suggestions:
        suggestions.append("This combination already aligns well with high-performing patterns in the data.")

    return suggestions


In [8]:
# 9. Interactive ipywidgets form
idea_text_box = widgets.Textarea(
    value="A minimalist white t-shirt for unisex, made from organic cotton, perfect for summer.",
    description="Idea:", layout=widgets.Layout(width="600px", height="80px"),
)

gender_dd = widgets.Dropdown(options=CATEGORY_OPTIONS["gender"], description="Gender:")
category_dd = widgets.Dropdown(options=CATEGORY_OPTIONS["category"], description="Category:")
subcategory_dd = widgets.Dropdown(options=CATEGORY_OPTIONS["subcategory"], description="Subcategory:")
fabric_dd = widgets.Dropdown(options=CATEGORY_OPTIONS["fabric"], description="Fabric:")
theme_dd = widgets.Dropdown(options=CATEGORY_OPTIONS["theme"], description="Theme:")
season_dd = widgets.Dropdown(options=CATEGORY_OPTIONS["season"], description="Season:")
occasion_dd = widgets.Dropdown(options=CATEGORY_OPTIONS["occasion"], description="Occasion:")
color_dd = widgets.Dropdown(options=CATEGORY_OPTIONS["color"], description="Color:")

price_slider = widgets.FloatSlider(value=1500, min=100, max=10000, step=50, description="Price:")
rating_slider = widgets.FloatSlider(value=4.0, min=1.0, max=5.0, step=0.1, description="Est. rating:")
review_slider = widgets.IntSlider(value=100, min=0, max=3000, step=10, description="Est. reviews:")

predict_btn = widgets.Button(description="Predict", button_style="success")
output_area = widgets.Output()


def on_predict_click(b):
    with output_area:
        clear_output()
        result = predict_idea(
            idea_text_box.value, gender_dd.value, category_dd.value, subcategory_dd.value,
            fabric_dd.value, theme_dd.value, season_dd.value, occasion_dd.value, color_dd.value,
            price_slider.value, rating_slider.value, review_slider.value,
        )

        print(f"Success score: {result['score']:.1f} / 100")
        print(f"Predicted label: {result['label'].upper()}")
        print("Probability breakdown:")
        for cls, p in result["proba"].items():
            print(f"  {cls}: {p*100:.1f}%")

        print("\nSuggestions:")
        for s in suggest_improvements(result, fabric_dd.value, season_dd.value, theme_dd.value, subcategory_dd.value):
            print(f"  - {s}")

        print("\nMost similar existing ideas:")
        display(result["similar"])

        fig, ax = plt.subplots(figsize=(6, 4))
        compare_scores = result["similar"][REG_TARGET].tolist() + [result["score"]]
        compare_labels = [f"match {i+1}" for i in range(len(result["similar"]))] + ["YOUR IDEA"]
        colors = ["#85B7EB"] * len(result["similar"]) + ["#D85A30"]
        ax.barh(compare_labels, compare_scores, color=colors)
        ax.set_xlabel("Success score")
        ax.set_title("Your idea vs. most similar existing ideas")
        plt.tight_layout()
        plt.show()


predict_btn.on_click(on_predict_click)

form = widgets.VBox([
    idea_text_box,
    widgets.HBox([gender_dd, category_dd, subcategory_dd]),
    widgets.HBox([fabric_dd, theme_dd, season_dd]),
    widgets.HBox([occasion_dd, color_dd]),
    price_slider, rating_slider, review_slider,
    predict_btn,
    output_area,
])
display(form)
